In [ ]:
import re
import json
import itertools
from datetime import datetime as dt

import boto3
from botocore.config import Config

import pandas as pd

## Create List of Products from Lookup

In [3]:
with open('../lookup/get_products.json', 'r') as lookup_get_products:
    lookup_dict = json.load(lookup_get_products)

In [4]:
product_lookup_list = []

for prd in lookup_dict['ProductFilters']:

    filters = {**prd, **lookup_dict['LocationFilters']}

    filters_as_lists = {
        k: [v] if not isinstance(v, list) else v
        for k, v
        in filters.items()
    }

    filters_all = [
        dict(zip(filters_as_lists.keys(), values))
        for values
        in itertools.product(*filters_as_lists.values())
    ]

    product_lookup_list += filters_all

## Create Messages from Lookup List

In [ ]:
message_list = []

for product in product_lookup_list:

    params = {
        'ServiceCode': product['servicecode'],
        'Filters':
        [
            { 'Type': 'TERM_MATCH', 'Field': key, 'Value': value }
            for key, value in product.items()
        ]
    }

    filepath_keys = ['location', 'servicecode', 'productFamily']
    filepath = ''

    for fp_key in filepath_keys:
        for key, value in product.items():
            if key == fp_key:
                filepath += re.sub(r'[\(\)\s-]', '', f"{value}/")

    instance_flag = 0

    for key, value in product.items():
        if key == 'productFamily' and value == 'Compute Instance':
            instance_flag = 1

    message_list.append({
        'params': params,
        'filepath': filepath,
        'instance_flag': instance_flag
    })

message_list = message_list[:3]

## Loop through get_products endpoint

In [66]:
CONFIG = Config(
   retries = {
      'max_attempts': 10,
      'mode': 'standard'
   }
)

session = boto3.Session(profile_name='psilv')
pricing_client = session.client('pricing', region_name='us-east-1', config=CONFIG)

In [ ]:
paginator = pricing_client.get_paginator('get_products')

product_list = []
all_pages = []

for m in message_list:

  page_iterator = paginator.paginate(
    ServiceCode=m['params']['ServiceCode'],
    Filters=m['params']['Filters'],
  )

  for page in page_iterator:
    product_list += page['PriceList']

product_list_parsed = [eval(p) for p in product_list]


## Output

### Write to S3

In [90]:
S3_BUCKET = 'ps-til-test-awspricelist'
S3_PREFIX = 'data/getproducts/'

filepath = message_list[0]['filepath']
filename = f'{dt.now().strftime('%Y-%m-%d-%H%M')}.json'

prefix = f'{S3_PREFIX}{filepath}'
key = f'{prefix}{filename}'

json_data = json.dumps(
    product_list_parsed,
    indent=4,
    ensure_ascii=False
)

s3_client = session.client('s3')
s3_response = s3_client.put_object(
    Body=json_data,
    Bucket=S3_BUCKET,
    Key=key,
)

### Write Product List to json

In [ ]:
# filename = f'./{dt.now().strftime('%Y-%m-%d')}.json'

# with open(filename, 'w') as json_file:
#     json.dump(product_list_parsed, json_file, indent=2)

## Old Code

### Misc

In [94]:
# m = message_list[0]

# page_iterator = paginator.paginate(
#   ServiceCode=m['params']['ServiceCode'],
#   Filters=m['params']['Filters']
# )


# message_list_short = message_list[:2]

# page_iterator_all = []
# for m in message_list_short:

#   page_iterator = paginator.paginate(
#     ServiceCode=m['params']['ServiceCode'],
#     Filters=m['params']['Filters']
#   )

#   page_iterator_all.append(page_iterator)

In [95]:
# product_list = []

# page_iterator = paginator.paginate(
#     ServiceCode=m['params']['ServiceCode'],
#     Filters=m['params']['Filters']
# )

# for page in page_iterator:
#     pages = page['PriceList']

# for page in page_iterator_all:
#     for item in page['PriceList']:
#         product_list.append(json.loads(item))

In [96]:
# pd.DataFrame.from_dict(pages).to_csv("pages.csv",  index=False)

### Create paginator for get_products

In [97]:
# CONFIG = Config(
#    retries = {
#       'max_attempts': 10,
#       'mode': 'standard'
#    }
# )

# session = boto3.Session(profile_name='psilv')
# pricing_client = session.client('pricing', region_name='us-east-1', config=CONFIG)

# paginator = pricing_client.get_paginator('get_products')

### Iterate through paginator

In [98]:
# message_list_short = message_list[:3]
# product_list = []

# for m in message_list_short:

#   page_iterator = paginator.paginate(
#     ServiceCode=m['params']['ServiceCode'],
#     Filters=m['params']['Filters']
#   )

#   for page in page_iterator:
#     item = page['PriceList']
#     product_list.append(json.dumps(item))
